In [ ]:
import os
os.environ['HF_TOKEN'] = "<hf_token>"

In [ ]:
import evaluate
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics.pairwise import cosine_similarity
import torch
import numpy as np
from transformers import set_seed
from tqdm import tqdm

In [ ]:
from datasets import load_dataset
import os
path =  os.path.dirname(os.path.dirname(os.getcwd()))
path = os.path.join(path, 'data')

seed = 42
set_seed(seed)


dataset_pt = load_dataset('json', data_files=path+'/ordered_PT_test_dataset.json')['train']
dataset_hc = load_dataset('json', data_files=path+'/ordered_healthy_test_dataset.json')['train']


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
system_prompt = "You are a patient that has gone to do an interview with a psychologist. The psychologist will ask you a series of questions and you will answer them in a natural way:\n"
user_prompt = "### Input:\n{question}\n\n### Expected Response:\n{answer}"

def apply_prompt(example):
    example["text"] = (
        system_prompt
        + user_prompt.format(question=example["question"], answer=example["answer"])
    )
    return example
dataset_pt = dataset_pt.map(apply_prompt)
dataset_hc = dataset_hc.map(apply_prompt)

Map:   0%|          | 0/778 [00:00<?, ? examples/s]

Map:   0%|          | 0/243 [00:00<?, ? examples/s]

In [ ]:
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
perplexity = evaluate.load("perplexity")

reales_pt = [i['text'] for i in dataset_pt]
reales_hc = [i['text'] for i in dataset_hc]

In [ ]:
def generate_responses(df,model,tokenizer):
    predictions = []

    reales_only_response = []
    output_only_response = []
    for real in tqdm(df):
        real_no_response = real[:real.find('### Expected Response:') + len("### Expected Response:")].strip()
        inputs = tokenizer(real_no_response, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=None,
                do_sample=True,
                temperature=0.5,
                top_p=0.90,
                pad_token_id=tokenizer.eos_token_id
            )
        output = tokenizer.decode(outputs[0], skip_special_tokens=True)
        reales_only_response.append(real[real.find('### Expected Response:') + len('### Expected Response:'):].strip())
        output_only_response.append(output[output.find('### Expected Response:') + len('### Expected Response:'):].strip())
        predictions.append(output)
    return predictions, reales_only_response, output_only_response
def calculate_perplexities(reales, model,tokenizer):
    perplexities = []
    model.eval()
    with torch.no_grad():
        for text in tqdm(reales, desc="Calculando perplexities",leave=False):
            enc = tokenizer(text, return_tensors="pt").to(model.device)
            outputs = model(**enc, labels=enc["input_ids"])
            loss = outputs.loss.item()  # loss media por token (cross-entropy)
            ppl = float(np.exp(loss))
            perplexities.append(ppl)
    return float(np.mean(perplexities))
def embed_texts(texts):
    encoded_input = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(model.device)
    with torch.no_grad():
        # activar salida de todos los hidden states para obtener embeddings
        outputs = model(**encoded_input, output_hidden_states=True)
        hidden_states = outputs.hidden_states
        last_hidden = hidden_states[-1]
        # promediar embeddings de tokens por cada secuencia (dim=1)
        embeddings = last_hidden.mean(dim=1)
        # convertir a float32 y mover a CPU para compatibilidad con sklearn/numpy
        embeddings = embeddings.to(torch.float32).cpu()
    return embeddings
def batch_semantic_similarity(reference_texts, generated_texts):
    similarities = []
    for i in tqdm(range(0,len(reference_texts),2), desc="Calculando similitudes semánticas"):
        ref_emb = embed_texts(reference_texts[i:i+2])
        gen_emb = embed_texts(generated_texts[i:i+2])
        # convertir a numpy antes de usar sklearn
        ref_np = [j.unsqueeze(0).numpy() for j in ref_emb]
        gen_np = [j.unsqueeze(0).numpy() for j in gen_emb]
        sim = [float(cosine_similarity(r, g)[0][0]) for r, g in zip(ref_np, gen_np)]
        similarities.extend(sim)
    return similarities



def calculate_values(df, model, tokenizer):
    predictions, reales_only_response,output_only_response = generate_responses(df,model, tokenizer)
    
    references_for_bleu = [[r] for r in reales_only_response]  

    bleu_result = bleu.compute(predictions=predictions, references=references_for_bleu)
    print("BLEU:", bleu_result["bleu"])
    
    rouge_result = rouge.compute(predictions=predictions, references=reales_only_response)
    print("ROUGE:", rouge_result)

    avg_perplexity = calculate_perplexities(df,model,tokenizer)
    print("Mean perplexity:", avg_perplexity )

    sim_scores = batch_semantic_similarity(reales_only_response, output_only_response)
    print('Semantic similarity average:', np.array(sim_scores).mean())



---
## Pacientes 
### model fine-tunned


In [10]:
model_name = "PabloCano1/ordered-PT-gemma3-4b-fine-tuned"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, attn_implementation="eager", device_map="cuda:0", dtype=torch.bfloat16
)
print("***Model fine-tuned psychosis***")
calculate_values(reales_pt,model, tokenizer)

model_name = "google/gemma-3-4b-it"
model = AutoModelForCausalLM.from_pretrained(
    model_name, attn_implementation="eager", device_map="cuda:0", dtype=torch.bfloat16
)
print("***Original model psychosis***")
calculate_values(reales_pt,model,tokenizer)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

***Model fine-tuned psychosis***


100%|████████████████████████████████████████████████████████████████████████| 778/778 [11:50<00:00,  1.10it/s]


BLEU: 0.018476929439343367
ROUGE: {'rouge1': np.float64(0.12671978102584283), 'rouge2': np.float64(0.014897705763447799), 'rougeL': np.float64(0.07976959003365282), 'rougeLsum': np.float64(0.11690130488335128)}


Mean perplexity: 3.447218045929522


...

100%|████████████████████████████████████████████████████████████████████████| 778/778 [16:12<00:00,  1.25s/it]


BLEU: 0.008857409275499037
ROUGE: {'rouge1': np.float64(0.11374073833782866), 'rouge2': np.float64(0.009716118222850803), 'rougeL': np.float64(0.07304510768586328), 'rougeLsum': np.float64(0.10445503893669444)}


Mean perplexity: 64.16664512334637


Calculando similitudes semánticas: 100%|█████████████████████████████████████| 389/389 [01:13<00:00,  5.30it/s]

Semantic similarity average: 0.5004974305282579


---
## Controles
### model fine-tunned

In [9]:
model_name = "PabloCano1/ordered-HC-gemma3-4b-fine-tuned"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, attn_implementation="eager", device_map="cuda:0", dtype=torch.bfloat16
)

print("***Model fine-tuned healthy control***")
calculate_values(reales_hc,model, tokenizer)

model_name = "google/gemma-3-4b-it"
model = AutoModelForCausalLM.from_pretrained(
    model_name, attn_implementation="eager", device_map="cuda:0", dtype=torch.bfloat16
)
print("***Original model healthy control***")
calculate_values(reales_hc,model, tokenizer)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

...

100%|████████████████████████████████████████████████████████████████████████| 243/243 [05:19<00:00,  1.31s/it]


BLEU: 0.009574923983112648
ROUGE: {'rouge1': np.float64(0.13001521390689397), 'rouge2': np.float64(0.01629827805556532), 'rougeL': np.float64(0.07955005640625028), 'rougeLsum': np.float64(0.12206779418904573)}


Mean perplexity: 49.29418509904351


Calculando similitudes semánticas: 100%|█████████████████████████████████████| 122/122 [00:24<00:00,  5.06it/s]

Semantic similarity average: 0.5091035940412263
